# YOLO11m 74-class 5-fold MPS + OHEM Parameter Crosscheck

Purpose:
- Check YOLO settings against the existing RF-DETR 5-fold setup before training.
- Verify class mapping, fold counts, background negatives, augmentation, output paths, and OHEM patch activation.
- Training is optional and disabled by default.

Path setup:
- `REPO_ROOT` is this git checkout.
- `YOLO_ROOT` is `REPO_ROOT / "YOLO"` and contains this notebook/tools.
- `DATA_ROOT` points to the workspace that owns `working/` datasets and outputs.
- Set env vars if code and data live in different folders. No machine-specific absolute paths are stored in the notebook.

Important:
- Background images are used as empty-label YOLO images, not as category-0 boxes.
- OHEM keeps all foreground anchors and only hard background anchors for classification loss.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Shared-path setup. Override with env vars if your checkout/data layout differs.
# REPO_ROOT: this git checkout root.
# YOLO_ROOT: folder containing this YOLO experiment code.
# DATA_ROOT: project/data workspace that owns working/ datasets and outputs.
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).resolve()
YOLO_ROOT = Path(os.environ.get('YOLO_ROOT', REPO_ROOT / 'YOLO')).resolve()
DATA_ROOT = Path(os.environ.get('DATA_ROOT', os.environ.get('PROJECT_ROOT', REPO_ROOT))).resolve()
PYTHON = Path(os.environ.get('PYTHON', sys.executable)).resolve()
RFDETR_REPO = Path(os.environ.get('RFDETR_REPO', REPO_ROOT.parent / 'ai12-team01-rfdetr')).resolve()

YOLO_DATASET = Path(os.environ.get('YOLO_DATASET', DATA_ROOT / 'working/yolo_74_5fold_bg_mps')).resolve()
YOLO_OUTPUT = Path(os.environ.get('YOLO_OUTPUT', DATA_ROOT / 'working/yolo_outputs/yolo11m_74_5fold_bg_mps_10ep')).resolve()
RFDETR_CONFIG = Path(os.environ.get(
    'RFDETR_CONFIG',
    RFDETR_REPO / 'RF_DETR_split_ver/config_74_hidden45_mps_rfdetr_large_v18_5fold_p00.yaml',
)).resolve()
RFDETR_FOLDS = Path(os.environ.get(
    'RFDETR_FOLDS',
    DATA_ROOT / 'working/rfdetr_dataset_74_hidden45_canvas_balanced_5fold_cls0_mps',
)).resolve()
REPORT_DIR = Path(os.environ.get(
    'YOLO_AUDIT_REPORT_DIR',
    DATA_ROOT / 'working/reports/yolo_rfdetr_param_audit_20260709',
)).resolve()
PREVIEW_IMAGE = Path(os.environ.get(
    'YOLO_PREVIEW_IMAGE',
    DATA_ROOT / 'working/reports/yolo_74_5fold_bg_mps_preview_20260709/fold0_train_yolo_labels_and_backgrounds_preview.jpg',
)).resolve()

MODEL = 'yolo11m.pt'
FOLDS = '0,1,2,3,4'
EPOCHS = 10
IMGSZ = 960
BATCH = 4
WORKERS = 2
DEVICE = 'mps'
SCALE_MIN = 0.9
SCALE_MAX = 1.0
BEST_METRIC = 'map75_95'
OHEM_NEGATIVE_RATIO = 0.25
OHEM_MIN_NEG = 16
OHEM_BG_NEG = 32
OHEM_NEG_WEIGHT = 0.25

RUN_INSTALL = False
RUN_PARAM_AUDIT = True
RUN_OHEM_DRY_RUN = True
RUN_TRAIN = False

print('REPO_ROOT:', REPO_ROOT)
print('YOLO_ROOT:', YOLO_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('PYTHON:', PYTHON)
print('RFDETR_REPO:', RFDETR_REPO)
print('YOLO_DATASET:', YOLO_DATASET)
print('YOLO_OUTPUT:', YOLO_OUTPUT)
print('RFDETR_CONFIG:', RFDETR_CONFIG)


In [ ]:
if RUN_INSTALL:
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-U', 'ultralytics', 'pyyaml', 'pandas'], check=True)

try:
    import ultralytics
    print('ultralytics:', ultralytics.__version__)
except Exception as e:
    print('Ultralytics import failed:', e)
    print('Set RUN_INSTALL=True and rerun this cell if needed.')


In [ ]:
# Parameter crosscheck: RF-DETR config vs YOLO setup.
if RUN_PARAM_AUDIT:
    cmd = [
        str(PYTHON), str(YOLO_ROOT / 'tools/audit_yolo_rfdetr_params.py'),
        '--rfdetr-config', str(RFDETR_CONFIG),
        '--rfdetr-folds', str(RFDETR_FOLDS),
        '--yolo-dataset', str(YOLO_DATASET),
        '--out', str(REPORT_DIR),
        '--yolo-model', MODEL,
        '--yolo-epochs', str(EPOCHS),
        '--yolo-imgsz', str(IMGSZ),
        '--yolo-batch', str(BATCH),
        '--yolo-workers', str(WORKERS),
        '--yolo-device', DEVICE,
        '--yolo-scale-min', str(SCALE_MIN),
        '--yolo-scale-max', str(SCALE_MAX),
        '--yolo-best-metric', BEST_METRIC,
    ]
    print('run:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image

summary_path = REPORT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(Markdown('## Audit Summary'))
    display(summary)

param_csv = REPORT_DIR / 'parameter_crosscheck.csv'
fold_csv = REPORT_DIR / 'fold_data_crosscheck.csv'
class_csv = REPORT_DIR / 'class_mapping_crosscheck.csv'

if param_csv.exists():
    display(Markdown('## Parameter Crosscheck'))
    display(pd.read_csv(param_csv))
if fold_csv.exists():
    display(Markdown('## Fold Data Crosscheck'))
    display(pd.read_csv(fold_csv))
if class_csv.exists():
    class_df = pd.read_csv(class_csv)
    display(Markdown(f'## Class Mapping Crosscheck: mismatches={int((~class_df.ok).sum())}'))
    display(class_df.head(10))
    bad = class_df[~class_df.ok]
    if len(bad):
        display(bad)


In [ ]:
# Visual sanity check: YOLO label conversion and background empty-label images.
if PREVIEW_IMAGE.exists():
    display(Image(filename=str(PREVIEW_IMAGE)))
else:
    print('Preview image not found:', PREVIEW_IMAGE)


In [ ]:
# OHEM dry-run: verifies that the runtime loss patch is active before training.
if RUN_OHEM_DRY_RUN:
    cmd = [
        str(PYTHON), str(YOLO_ROOT / 'tools/train_yolo_5fold_mps.py'),
        '--dry-run',
        '--require-ohem',
        '--model', MODEL,
        '--folds', '0',
        '--epochs', str(EPOCHS),
        '--imgsz', str(IMGSZ),
        '--batch', str(BATCH),
        '--workers', str(WORKERS),
        '--device', DEVICE,
        '--scale-min', str(SCALE_MIN),
        '--scale-max', str(SCALE_MAX),
        '--ohem-negative-ratio', str(OHEM_NEGATIVE_RATIO),
        '--ohem-min-neg', str(OHEM_MIN_NEG),
        '--ohem-bg-neg', str(OHEM_BG_NEG),
        '--ohem-neg-weight', str(OHEM_NEG_WEIGHT),
    ]
    print('run:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## Training

Training is disabled by default. Set `RUN_TRAIN = True` in the first parameter cell after the audit tables look sane.

Default run:
- `yolo11m.pt`
- 5 folds
- 10 epochs
- best checkpoint selected by mAP@0.75:0.95 (`best_map75_95.pt`)
- scale augmentation 0.9~1.0
- MPS
- batch 4, workers 2
- OHEM required
- background empty-label negatives enabled in train only
- mosaic disabled (`mosaic=0.0`) so scale stays constrained
- OHEM defaults: negative_ratio=0.25, min_neg=16, bg_neg=32, neg_weight=0.25


In [ ]:
if RUN_TRAIN:
    cmd = [
        str(PYTHON), '-u', str(YOLO_ROOT / 'tools/train_yolo_5fold_mps.py'),
        '--require-ohem',
        '--model', MODEL,
        '--folds', FOLDS,
        '--epochs', str(EPOCHS),
        '--imgsz', str(IMGSZ),
        '--batch', str(BATCH),
        '--workers', str(WORKERS),
        '--device', DEVICE,
        '--scale-min', str(SCALE_MIN),
        '--scale-max', str(SCALE_MAX),
        '--ohem-negative-ratio', str(OHEM_NEGATIVE_RATIO),
        '--ohem-min-neg', str(OHEM_MIN_NEG),
        '--ohem-bg-neg', str(OHEM_BG_NEG),
        '--ohem-neg-weight', str(OHEM_NEG_WEIGHT),
    ]
    print('run:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
else:
    print('RUN_TRAIN=False. Audit only; no training launched.')


In [ ]:
# After training: summarize fold result files if they exist.
result_rows = []
for fold_dir in sorted(YOLO_OUTPUT.glob('fold*')):
    results_csv = fold_dir / 'results.csv'
    best_pt = fold_dir / 'weights/best.pt'
    last_pt = fold_dir / 'weights/last.pt'
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        tail = df.tail(1).to_dict('records')[0] if len(df) else {}
        result_rows.append({
            'fold': fold_dir.name,
            'results_csv': str(results_csv),
            'best_pt_exists': best_pt.exists(),
            'last_pt_exists': last_pt.exists(),
            **tail,
        })
if result_rows:
    display(pd.DataFrame(result_rows))
else:
    print('No YOLO training results found yet:', YOLO_OUTPUT)


## Post-training RF-DETR/YOLO Result Crosscheck (later)

After YOLO weights exist, add/use a result comparison cell that:
1. runs YOLO inference on the original test set,
2. maps YOLO class ids back through `label_map_yolo74.csv`,
3. compares positions/classes with RF-DETR WBF output by IoU,
4. exports mismatch crops and CSV.

This notebook intentionally does not run result comparison before YOLO training exists.
